# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
import subprocess
import sys

# Este código fuerza a Python a instalar las librerías internamente
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sqlalchemy", "jupysql", "psycopg2-binary"])
    print("¡Librerías instaladas con éxito! Ahora reinicia el Kernel.")
    
except Exception as e:
    print(f"Error al instalar: {e}")

¡Librerías instaladas con éxito! Ahora reinicia el Kernel.


In [2]:
import subprocess
import sys

try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "psycopg[binary]"])
    print("¡Driver psycopg instalado! Recuerda ir a Kernel -> Restart.")
except Exception as e:
    print(f"Error: {e}")

¡Driver psycopg instalado! Recuerda ir a Kernel -> Restart.


In [3]:
import subprocess
import sys

try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels"])
    print("¡Librería statsmodels instalada con éxito!")
except Exception as e:
    print(f"Error al instalar: {e}")


¡Librería statsmodels instalada con éxito!


In [4]:
# importar librerías
import pandas as pd
import numpy as np

In [5]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [6]:
# Exploración estadística para detectar los outliers (Valores atípicos)
columnas_numericas = ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']

print("--- Resumen Estadístico de Órdenes (Datos Crudos) ---")
display(orders[columnas_numericas].describe())

print("--- Resumen Estadístico de Catálogo (Numéricas) ---")
display(catalog.describe())

print("\n--- Resumen Estadístico de Marketing (Numéricas) ---")
display(marketing.describe())

print("\n--- Resumen Estadístico de Catálogo (Categóricas) ---")
display(catalog.describe(include=['object','string']))

print("\n--- Resumen Estadístico de Marketing (Categóricas) ---")
display(marketing.describe(include=['object','string']))

--- Resumen Estadístico de Órdenes (Datos Crudos) ---


,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


--- Resumen Estadístico de Catálogo (Numéricas) ---


,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000



--- Resumen Estadístico de Marketing (Numéricas) ---


,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000



--- Resumen Estadístico de Catálogo (Categóricas) ---


,nombre_producto,categoria_producto,proveedor
count,7,7,7
unique,7,3,7
top,Laptop-Gaming-16GB,Electrónica,"Fuller, Pena and Myers"
freq,1,3,1



--- Resumen Estadístico de Marketing (Categóricas) ---


,fecha,pais,id_campaña,canal
count,1620,1620,1620,1519
unique,180,3,9,3
top,2025-01-01,Mexico,organic_Mexico,paid_search
freq,9,540,180,507


In [7]:
# explorar datasets
print("--- Orders ---")
orders.info()
display(orders.sample(25))


--- Orders ---
<class 'pandas.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  str    
 1   id_usuario          25100 non-null  str    
 2   fecha_hora_pedido   25100 non-null  str    
 3   pais                24800 non-null  str    
 4   dispositivo         25080 non-null  str    
 5   fuente_referencia   25070 non-null  str    
 6   nombre_producto     25070 non-null  str    
 7   categoria_producto  25020 non-null  str    
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), str(8)
memory usage: 2.3 MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
18495,order_18495,user_5935,2025-05-28,Argentina,desktop,organic,Sneakers-Urban-42,Moda,2.0,177.99,0.0,355.98
7578,order_7578,user_6069,2025-05-31,Mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,1.0,29.74,0.0,29.74
3417,order_3417,user_4434,2025-05-01,Mexico,mobile,social,Phone-Pro-128GB,Electronica,2.0,105.42,15.0,195.84
8516,order_8516,user_4281,2025-04-15,Argentina,mobile,paid_search,Jacket-Winter-M,Moda,2.0,289.27,10.0,568.54
18175,order_18175,user_507,2025-04-07,mexico,desktop,social,Tablet-Standard-64GB,Electronica,2.0,256.73,0.0,513.47
16444,order_16444,user_6389,2025-04-04,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,212.01,5.0,419.02
11966,order_11966,user_118,2025-01-01,Argentina,desktop,social,Sneakers-Urban-42,Moda,1.0,497.47,0.0,497.47
7312,order_7312,user_3190,2025-05-18,Argentina,desktop,social,Vacuum-Pro-Black,Hogar,1.0,63.65,15.0,48.65
11541,order_11541,user_1438,2025-04-29,Argentina,mobile,paid_search,Sneakers-Urban-42,Moda,1.0,105.74,5.0,100.74
15617,order_15617,user_2087,2025-03-22,Colombia,mobile,organic,Jacket-Winter-M,Moda,2.0,74.81,0.0,149.62


In [8]:
print("\n--- Catalog ---")
catalog.info()
display(catalog.head(7))


--- Catalog ---
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      str    
 1   categoria_producto  7 non-null      str    
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      str    
dtypes: float64(1), str(3)
memory usage: 356.0 bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [9]:
print("\n--- Marketing ---")
marketing.info()
display(marketing.head())


--- Marketing ---
<class 'pandas.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   str    
 1   pais        1620 non-null   str    
 2   id_campaña  1620 non-null   str    
 3   canal       1519 non-null   str    
 4   gasto       1620 non-null   float64
dtypes: float64(1), str(4)
memory usage: 63.4 KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [10]:
# tu código aquí
#1. Convertir fechas
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')

In [11]:
# 2. Eliminar duplicados
orders = orders.drop_duplicates().reset_index(drop=True)
catalog = catalog.drop_duplicates().reset_index(drop=True)
marketing = marketing.drop_duplicates().reset_index(drop=True)

In [12]:
# 3. Regla de IQR para filtrar outliers de cantidad
Q1 = orders['cantidad'].quantile(0.25)
Q3 = orders['cantidad'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

# 4. Filtrar el dataset (Valores válidos + IQR)
orders_clean = orders[(orders['cantidad'] > 0) & (orders['cantidad'] <= limite_superior) & (orders['precio_unitario'] > 0)].copy()

# Eliminar nulos críticos
orders_clean = orders_clean.dropna(subset=['nombre_producto', 'precio_unitario'])

In [13]:
# 5. Rellenar nulos en descuento y categóricas
orders_clean['monto_descuento'] = orders_clean['monto_descuento'].fillna(0)
for col in ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']:
    orders_clean[col] = orders_clean[col].fillna('unknown')

# Estandarizar texto en Orders (quitar espacios y homologar mayúsculas/minúsculas)
orders_clean['nombre_producto'] = orders_clean['nombre_producto'].str.strip()
orders_clean['pais'] = orders_clean['pais'].str.strip().str.title()
orders_clean['pais'] = orders_clean['pais'].replace('Unknown', 'unknown')

orders_clean['categoria_producto'] = orders_clean['categoria_producto'].str.strip().str.title()
orders_clean['categoria_producto'] = orders_clean['categoria_producto'].replace({'Electronica': 'Electrónica', 'Unknown': 'unknown'})

for col in ['dispositivo', 'fuente_referencia']:
    orders_clean[col] = orders_clean[col].str.strip().str.lower()

# Estandarizar texto en Catalog
catalog['nombre_producto'] = catalog['nombre_producto'].str.strip()
catalog['categoria_producto'] = catalog['categoria_producto'].str.strip().str.title()
catalog['proveedor'] = catalog['proveedor'].str.strip()

# 6. Limpieza y estandarización de Marketing
marketing_clean = marketing[marketing['gasto'] >= 0].copy()
marketing_clean['canal'] = marketing_clean['canal'].fillna('unknown').str.strip().str.lower()
marketing_clean['pais'] = marketing_clean['pais'].str.strip().str.title()
marketing_clean['id_campaña'] = marketing_clean['id_campaña'].str.strip()

print(f"Límite superior exacto por IQR para cantidad: {limite_superior}")
print(f"Filas originales: {orders.shape[0]} | Filas limpias: {orders_clean.shape[0]}")
print("Países únicos estandarizados en Orders:", orders_clean['pais'].unique())

Límite superior exacto por IQR para cantidad: 3.5
Filas originales: 25000 | Filas limpias: 24906
Países únicos estandarizados en Orders: <StringArray>
['Argentina', 'Mexico', 'Colombia', 'unknown']
Length: 4, dtype: str


In [14]:
# 7. Verificar y asegurar la consistencia de montos (usando orders_clean en toda la fórmula)
orders_clean['monto_total'] = (orders_clean['cantidad'] * orders_clean['precio_unitario']) - orders_clean['monto_descuento']

# Validación final
print("Nulos restantes en Orders:\n", orders_clean.isna().sum())
print("\nNulos restantes en Marketing:\n", marketing_clean.isna().sum())
print("\nTamaño final - Orders:", orders_clean.shape, " | Marketing:", marketing_clean.shape, " | Catalog:", catalog.shape)

Nulos restantes en Orders:
 id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64

Nulos restantes en Marketing:
 fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64

Tamaño final - Orders: (24906, 12)  | Marketing: (1620, 5)  | Catalog: (7, 4)


In [15]:
print("--- Auditoría final de formato de datos ---")
print("\n--- Catalog ---")
catalog.info()
display(catalog.head(7))

print("\n--- Marketing ---")
marketing_clean.info()
display(marketing_clean.head(7))

print("\n--- Orders ---")
orders_clean.info()
display(orders_clean.head(7))

--- Auditoría final de formato de datos ---

--- Catalog ---
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      str    
 1   categoria_producto  7 non-null      str    
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      str    
dtypes: float64(1), str(3)
memory usage: 356.0 bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes



--- Marketing ---
<class 'pandas.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[us]
 1   pais        1620 non-null   str           
 2   id_campaña  1620 non-null   str           
 3   canal       1620 non-null   str           
 4   gasto       1620 non-null   float64       
dtypes: datetime64[us](1), float64(1), str(3)
memory usage: 63.4 KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40
5,2025-01-01,Colombia,social_Colombia,social,833.38
6,2025-01-01,Argentina,organic_Argentina,organic,1365.62



--- Orders ---
<class 'pandas.DataFrame'>
Index: 24906 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24906 non-null  str           
 1   id_usuario          24906 non-null  str           
 2   fecha_hora_pedido   24906 non-null  datetime64[us]
 3   pais                24906 non-null  str           
 4   dispositivo         24906 non-null  str           
 5   fuente_referencia   24906 non-null  str           
 6   nombre_producto     24906 non-null  str           
 7   categoria_producto  24906 non-null  str           
 8   cantidad            24906 non-null  float64       
 9   precio_unitario     24906 non-null  float64       
 10  monto_descuento     24906 non-null  float64       
 11  monto_total         24906 non-null  float64       
dtypes: datetime64[us](1), float64(4), str(7)
memory usage: 2.5 MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.38
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electrónica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.98
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electrónica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28
5,order_5,user_2792,2025-04-22,Mexico,desktop,organic,Tablet-Standard-64GB,Electrónica,1.0,179.34,0.0,179.34
6,order_6,user_2802,2025-03-15,Colombia,mobile,organic,Blender-XL-Red,Hogar,2.0,163.59,0.0,327.18


Estructura de la limpieza:

- Fechas seguras: Forzamos la conversión a datetime para poder agrupar ventas por mes más adelante.

- Formato de categorías en columnas: Se corrigen los formatos para quesean consistentes en todas las filas, quitando espacios extras y guardando todos los paises iniciando con mayúscula.

- Filtro estricto financiero: Si cantidad o precio_unitario son nulos, negativos o cero, esos pedidos arruinarían KPIs de rentabilidad, por lo que se descartan.

- Salvavidas categórico: En lugar de borrar las 101 filas de marketing que no tienen canal, las etiquetamos como unknown. El dinero de esas campañas sí se gastó y debe sumarse al costo total para calcular cifras verdaderas sobre el profit.

- Recálculo de consistencia: Aseguramos que la columna monto_total sea el reflejo matemático exacto del precio por la cantidad menos el descuento, matando cualquier error tipográfico de la base cruda.

- Outliers: Se detectan ordenes con cantidad de productos vendidos como 10,000 y 20,000, aunque las cifras cuadran matemáticamente, estos valores son outliers por lo que se excluyen del análisis posterior para poder tener una confianza mayor en los datos.

In [16]:
# 1. Conciliación del impacto del filtro de limpieza (Calidad vs. Outliers IQR)
ordenes_invalidas = orders[(orders['cantidad'] <= 0) | (orders['precio_unitario'] <= 0) | (orders['nombre_producto'].isna()) | (orders['precio_unitario'].isna())]
outliers_iqr = orders[(orders['cantidad'] > limite_superior) & (orders['precio_unitario'] > 0) & (orders['nombre_producto'].notna())]

print("--- Conciliación de Registros Excluidos ---")
print(f"Registros tras eliminar duplicados exactos: {len(orders):,}")
print(f"Pedidos descartados por errores críticos (nulos/negativos/ceros): {len(ordenes_invalidas):,}")
print(f"Pedidos descartados por regla estadística IQR (cantidad > {limite_superior}): {len(outliers_iqr):,}")
print(f"Revenue nominal de los outliers excluidos: ${outliers_iqr['monto_total'].sum():,.2f}")
print(f"Total de pedidos válidos conservados: {len(orders_clean):,}\n")

# 2. Preservar trazabilidad del monto_total original antes de recalcular
orders_clean['monto_total_original'] = orders_clean['monto_total']
orders_clean['monto_total_calculado'] = (orders_clean['cantidad'] * orders_clean['precio_unitario']) - orders_clean['monto_descuento']
orders_clean['discrepancia_monto'] = (orders_clean['monto_total_original'] - orders_clean['monto_total_calculado']).round(2)

# Auditar discrepancias y asegurar que ningún descuento supere el monto bruto
discrepancias = orders_clean[orders_clean['discrepancia_monto'].abs() > 0.01]
montos_negativos = orders_clean[orders_clean['monto_total_calculado'] <= 0]

print(f"Órdenes con discrepancia > $0.01 entre monto original y recalculado: {len(discrepancias):,}")
print(f"Órdenes con monto final <= 0 tras aplicar descuento: {len(montos_negativos):,}")

# Asignar el monto verificado para el análisis financiero
orders_clean['monto_total'] = orders_clean['monto_total_calculado']

--- Conciliación de Registros Excluidos ---
Registros tras eliminar duplicados exactos: 25,000
Pedidos descartados por errores críticos (nulos/negativos/ceros): 84
Pedidos descartados por regla estadística IQR (cantidad > 3.5): 10
Revenue nominal de los outliers excluidos: $42,344,700.00
Total de pedidos válidos conservados: 24,906

Órdenes con discrepancia > $0.01 entre monto original y recalculado: 0
Órdenes con monto final <= 0 tras aplicar descuento: 0


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.
(Se exportan las versiones limpias "PBI" para que corran en Power BI con la notación correcta en las cifras).

In [17]:
# Exportar versiones para Power BI (con coma decimal) y para Python (con index=False)
orders_clean.to_csv('orders_cleanPBI.csv', index=False, decimal=',')
catalog.to_csv('catalog_cleanPBI.csv', index=False, decimal=',')
marketing_clean.to_csv('marketing_cleanPBI.csv', index=False, decimal=',')

orders_clean.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing_clean.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [18]:
# tu código aquí
import pandas as pd

In [19]:
# 1. Cargar los datasets limpios
orders_clean = pd.read_csv('orders_clean.csv')
catalog_clean = pd.read_csv('catalog_clean.csv')
marketing_clean = pd.read_csv('marketing_clean.csv')

# 2. Preparar 'orders_full' cruzando la tabla limpia con el catálogo
orders_full = orders_clean.merge(catalog_clean[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')

# 3. Asegurar costos y calcular el costo total (sin alterar el monto_total de origen)
orders_full['costo_unitario'] = orders_full['costo_unitario'].fillna(0)
orders_full['costo_total_producto'] = orders_full['costo_unitario'] * orders_full['cantidad']

# ==========================================
# Parte 1: Rentabilidad del negocio
# ==========================================

# 1. Ingreso total (Revenue) - Usamos la columna original
ingreso_total = orders_clean['monto_total'].sum()

# 2. Costo total de los productos
costo_productos = orders_full['costo_total_producto'].sum()

# 3. Inversión total en marketing
gasto_marketing = marketing_clean['gasto'].sum()

# 4. Rentabilidad (Profit)
profit_total = ingreso_total - costo_productos - gasto_marketing

print("--- Parte 1: Rentabilidad del negocio (Datos Limpios) ---")
print(f"Ingreso Total (Revenue): ${ingreso_total:,.2f}")
print(f"Costo Total de Productos: ${costo_productos:,.2f}")
print(f"Gasto Total en Marketing: ${gasto_marketing:,.2f}")
print(f"Ganancia Total (Profit): ${profit_total:,.2f}")

if profit_total > 0:
    print("\n ¡El modelo RappiPlus está generando ganancias netas!")
else:
    print("\n Alerta: El modelo RappiPlus está operando con pérdidas.")

--- Parte 1: Rentabilidad del negocio (Datos Limpios) ---
Ingreso Total (Revenue): $9,610,019.78
Costo Total de Productos: $3,828,869.01
Gasto Total en Marketing: $2,871,843.53
Ganancia Total (Profit): $2,909,307.24

 ¡El modelo RappiPlus está generando ganancias netas!


In [20]:
# Validar integridad del cruce entre pedidos y catálogo antes de calcular costos
filas_antes = len(orders_clean)
duplicados_catalogo = catalog_clean['nombre_producto'].duplicated().sum()

orders_full = orders_clean.merge(catalog_clean[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')
costos_faltantes = orders_full['costo_unitario'].isna().sum()

print("--- Auditoría de Integridad del Merge (Orders + Catalog) ---")
print(f"Filas antes del merge: {filas_antes:,} | Filas después del merge: {len(orders_full):,}")
print(f"Productos duplicados en catálogo: {duplicados_catalogo}")
print(f"Pedidos sin coincidencia de costo unitario (NaN): {costos_faltantes}\n")

# Calcular costo total una vez comprobado que hay 0 costos faltantes
orders_full['costo_total_producto'] = orders_full['costo_unitario'] * orders_full['cantidad']

--- Auditoría de Integridad del Merge (Orders + Catalog) ---
Filas antes del merge: 24,906 | Filas después del merge: 24,906
Productos duplicados en catálogo: 0
Pedidos sin coincidencia de costo unitario (NaN): 0



In [21]:
# ==========================================
# Parte 2: Comportamiento de ventas
# ==========================================

# 1. Ticket promedio por orden
ticket_promedio = orders_full['monto_total'].mean()

# 2. Cantidad promedio de productos por orden
cantidad_promedio = orders_full['cantidad'].mean()

# 3. Producto más vendido
producto_mas_vendido = orders_full.groupby('nombre_producto')['cantidad'].sum().idxmax()
unidades_vendidas = orders_full.groupby('nombre_producto')['cantidad'].sum().max()

# 4. Gasto en marketing por canal
gasto_por_canal = marketing_clean.groupby('canal')['gasto'].sum().sort_values(ascending=False)

print("\n--- Parte 2: Comportamiento de ventas (Datos Limpios) ---")
print(f"Ticket Promedio por orden: ${ticket_promedio:,.2f}")
print(f"Cantidad Promedio por orden: {cantidad_promedio:,.2f} unidades")
print(f"Producto más vendido: {producto_mas_vendido} (con {unidades_vendidas:,.0f} unidades)")
print("\nDesglose de Gasto en Marketing por Canal:")
print(gasto_por_canal)


--- Parte 2: Comportamiento de ventas (Datos Limpios) ---
Ticket Promedio por orden: $385.85
Cantidad Promedio por orden: 1.50 unidades
Producto más vendido: Vacuum-Pro-Black (con 6,284 unidades)

Desglose de Gasto en Marketing por Canal:
canal
social         918043.21
organic        913533.01
paid_search    863088.21
unknown        177179.10
Name: gasto, dtype: float64


Estructura del código:

Merge: Es indispensable cruzar orders con catalog usando nombre_producto. Sin esto, sabríamos cuánto se vendió, pero no cuánto le costó a Rappi adquirir esos productos para entregarlos.

Profit Total: En los negocios digitales, no basta con restar el costo del producto. También debemos restar el costo de adquisición de clientes (marketing) para obtener el beneficio neto (Profit).

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [22]:
import pandas as pd

In [23]:

from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [24]:
# Explorar tabla events
# =========================
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    GROUP BY nombre_evento
)
SELECT *
FROM funnel
ORDER BY usuarios_unicos DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [25]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    GROUP BY nombre_evento
),
ordered_funnel AS (
    SELECT
        nombre_evento,
        usuarios_unicos,
        LAG(usuarios_unicos) OVER (ORDER BY usuarios_unicos DESC) AS paso_anterior
    FROM funnel
)
SELECT
    nombre_evento,
    usuarios_unicos,
    ROUND((usuarios_unicos * 100.0) / paso_anterior, 2) AS conversion_paso_a_paso,
    ROUND((usuarios_unicos * 100.0) / FIRST_VALUE(usuarios_unicos) OVER (ORDER BY usuarios_unicos DESC), 2) AS conversion_total
FROM ordered_funnel
ORDER BY usuarios_unicos DESC;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,conversion_paso_a_paso,conversion_total
0,first_visit,7796,NaN,100.00
1,add_to_cart,7634,97.92,97.92
2,select_item,7582,99.32,97.26
3,begin_checkout,7208,95.07,92.46
4,add_payment_info,6250,86.71,80.17
5,purchase,6240,99.84,80.04


In [26]:
# PARTE 2: CORRECCIÓN --- Conversiones respetando la secuencia funcional del proceso de compra
query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS orden_funcional
    FROM events
    GROUP BY nombre_evento
),
ordered_funnel AS (
    SELECT
        orden_funcional,
        nombre_evento,
        usuarios_unicos,
        LAG(usuarios_unicos) OVER (ORDER BY orden_funcional) AS paso_anterior,
        FIRST_VALUE(usuarios_unicos) OVER (ORDER BY orden_funcional) AS usuarios_iniciales
    FROM funnel
)
SELECT
    orden_funcional,
    nombre_evento,
    usuarios_unicos,
    ROUND((usuarios_unicos * 100.0) / paso_anterior, 2) AS ratio_etapa_previa_pct,
    ROUND((usuarios_unicos * 100.0) / usuarios_iniciales, 2) AS proporcion_vs_inicio_pct
FROM ordered_funnel
ORDER BY orden_funcional;
'''

conversion = pd.read_sql(query_conversion, con=engine)
print("Funnel de Conversión por orden funcional")
display(conversion)

Funnel de Conversión por orden funcional


,orden_funcional,nombre_evento,usuarios_unicos,ratio_etapa_previa_pct,proporcion_vs_inicio_pct
0,1,first_visit,7796,NaN,100.00
1,2,select_item,7582,97.26,97.26
2,3,add_to_cart,7634,100.69,97.92
3,4,begin_checkout,7208,94.42,92.46
4,5,add_payment_info,6250,86.71,80.17
5,6,purchase,6240,99.84,80.04


**Conclusiones del Funnel de Conversión (Enfoque Secuencial vs. Presencia en Eventos) --- Correcciones**

* **Proporción Global de Compra (80.04%):** Al comparar los 6,240 usuarios únicos con evento `purchase` frente a los 7,796 registrados en `first_visit`, se observa una proporción de conversión aparente del 80.04%. Cabe precisar que este indicador mide la presencia de usuarios únicos por evento y no garantiza por sí solo una conversión estrictamente secuencial (donde cada usuario atraviesa todas las etapas previas en orden cronológico).
* **Anomalía en Navegación (`add_to_cart` > `select_item`):** Al ordenar el embudo según su secuencia lógica funcional, `add_to_cart` (7,634 usuarios) supera a `select_item` (7,582 usuarios), generando un ratio intermedio de **100.69%**. Esto sugiere como hipótesis que la interfaz permite agregar artículos directamente desde el listado general sin abrir la ficha del producto, o bien que existe una diferencia de instrumentación en el registro del evento `select_item`.
* **Principal Punto de Fricción (`begin_checkout` → `add_payment_info`):** La mayor caída relativa del recorrido ocurre al pasar del inicio de checkout (7,208 usuarios) al registro de información de pago (6,250 usuarios), reteniendo al **86.71%** (una contracción del 13.29%). Esto posiciona a la etapa de pago como la prioridad principal para investigar posibles fricciones de formulario, validación o métodos de pago.

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [27]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users
LIMIT 5;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)
display (users)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free
3,user_3,2025-03-04,Mexico,desktop,free
4,user_4,2025-02-27,Argentina,desktop,free


In [28]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''

SELECT *
FROM user_activity
LIMIT 5;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)
display(user_activity)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


In [29]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT 
        id_usuario,
        TO_CHAR(DATE_TRUNC('month', CAST(fecha_registro AS DATE)), 'YYYY-MM') AS cohorte
    FROM users
),
retencion AS (
    SELECT 
        c.cohorte,
        COUNT(DISTINCT c.id_usuario) AS clientes_iniciales,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 7 AND ua.activo = 1 THEN c.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 14 AND ua.activo = 1 THEN c.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN ua.dias_despues_registro = 21 AND ua.activo = 1 THEN c.id_usuario END) AS retenido_w3
    FROM cohortes c
    LEFT JOIN user_activity ua 
        ON c.id_usuario = ua.id_usuario
    GROUP BY c.cohorte
)
SELECT 
    cohorte,
    clientes_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND((retenido_w1 * 100.0) / NULLIF(clientes_iniciales, 0), 2) AS semana_1,
    ROUND((retenido_w2 * 100.0) / NULLIF(clientes_iniciales, 0), 2) AS semana_2,
    ROUND((retenido_w3 * 100.0) / NULLIF(clientes_iniciales, 0), 2) AS semana_3
FROM retencion
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
display(cohorte_final)

,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02,1444,611,609,635,42.31,42.17,43.98
2,2025-03,1636,677,705,690,41.38,43.09,42.18
3,2025-04,1606,680,697,663,42.34,43.40,41.28
4,2025-05,1687,695,676,706,41.20,40.07,41.85


In [30]:
# Retención acumulada por cohortes (según umbrales semanales >= 1, >= 2, >= 3)
query_cohort_retention_acumulada = '''
WITH cohortes AS (
    SELECT 
        id_usuario,
        TO_CHAR(DATE_TRUNC('month', CAST(fecha_registro AS DATE)), 'YYYY-MM') AS cohorte
    FROM users
),
actividad_semanal AS (
    SELECT
        id_usuario,
        FLOOR(dias_despues_registro / 7.0) AS semana_num
    FROM user_activity
    WHERE activo = 1 AND dias_despues_registro >= 7
),
retencion AS (
    SELECT 
        c.cohorte,
        COUNT(DISTINCT c.id_usuario) AS clientes_iniciales,
        COUNT(DISTINCT CASE WHEN a.semana_num >= 1 THEN c.id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN a.semana_num >= 2 THEN c.id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN a.semana_num >= 3 THEN c.id_usuario END) AS retenido_w3
    FROM cohortes c
    LEFT JOIN actividad_semanal a 
        ON c.id_usuario = a.id_usuario
    GROUP BY c.cohorte
)
SELECT 
    cohorte,
    clientes_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND((retenido_w1 * 100.0) / NULLIF(clientes_iniciales, 0), 2) AS semana_1,
    ROUND((retenido_w2 * 100.0) / NULLIF(clientes_iniciales, 0), 2) AS semana_2,
    ROUND((retenido_w3 * 100.0) / NULLIF(clientes_iniciales, 0), 2) AS semana_3
FROM retencion
ORDER BY cohorte;
'''

cohorte_acumulada = pd.read_sql(query_cohort_retention_acumulada, con=engine)
print("Cohorte con retencion semanal acumulada")
display(cohorte_acumulada)

Cohorte con retencion semanal acumulada


,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01,1627,1381,1253,1027,84.88,77.01,63.12
1,2025-02,1444,1255,1154,940,86.91,79.92,65.10
2,2025-03,1636,1428,1306,1060,87.29,79.83,64.79
3,2025-04,1606,1394,1261,1022,86.80,78.52,63.64
4,2025-05,1687,1446,1321,1088,85.71,78.30,64.49


In [31]:
# Exportar la tabla de cohortes de SQL para Power BI
cohorte_final.to_csv('cohortes_sql_PBI.csv', index=False, decimal=',')

### Consulta por Cohortes
Esta consulta agrupa a los usuarios basándose en su mes de registro (creando así las **cohortes**). Luego, cruza esa información con su registro de actividad (LEFT JOIN) para contar cuántos de esos usuarios originales (clientes_iniciales) tuvieron alguna interacción exactamente 7, 14 y 21 días después de haberse registrado. Finalmente, convierte esos conteos absolutos en porcentajes para poder comparar la retención semana a semana de forma estandarizada.

---

### Análisis de los Resultados de Retención

Al observar la matriz de cohortes, destacan los siguientes hallazgos clave para RappiPlus:

*   **Curva de retención inusualmente plana (Fidelidad del usuario activo):** A diferencia de muchos servicios donde la retención cae en picada semana tras semana, en RappiPlus la retención se estabiliza de inmediato. Los porcentajes entre semanas 1 a 3 se mantienen prácticamente iguales (fluctuando entre el **40% y el 43%**). Esto nos deja ver que, quien regresa la primera semana, adopta el servicio como un hábito continuo.
*   **Comportamiento estable mes a mes:** No existen saltos drásticos entre los usuarios adquiridos de enero a mayo de 2025. El volumen de adquisición se mantiene entre los 1,444 y 1,687 clientes iniciales, y todos los meses logran estabilizar su retención cerca del 42%.
*   **El gran desafío de la fuga temprana:** Aunque la retención a largo plazo es excepcionalmente estable, el punto crítico ocurre antes de la primera semana. Alrededor del **58%** de los usuarios no regresa para la semana 1. 

**Implicación de Negocio:** 
El servicio RappiPlus retiene exitosamente a su "núcleo duro" de usuarios. Las estrategias de crecimiento no deben enfocarse en retener a los usuarios de la semana 3, sino en mejorar el *Onboarding* (la experiencia de los primeros 5 días) para lograr que ese 58% de usuarios que abandona tempranamente experimente el valor del servicio y cruce la barrera de la primera semana.

### Análisis de Retención: Actividad Puntual vs. Retención Acumulada --- Correcciones de revisión

* **Distinción metodológica:** Al evaluar la actividad puntual (usuarios activos exactamente en los días 7, 14 y 21), las tasas semanales se mantienen estables entre el **40% y el 44%**. No obstante, esta estabilidad agregada no implica necesariamente que sean siempre los mismos individuos quienes interactúan cada semana (por ejemplo, en la cohorte de febrero la actividad puntual sube de 42.31% en la semana 1 a 43.98% en la semana 3, reflejando usuarios intermitentes que se reactivan en semanas posteriores). Al medir la **retención acumulada** (`semana_num >= 1, 2, 3`), observamos conjuntos progresivamente contenidos que muestran la proporción total de la cohorte que sigue presentando vida útil desde cada umbral semanal en adelante.
* **Inactividad temprana e hipótesis de Onboarding:** En ambas mediciones se identifica que una proporción importante de usuarios registrados no presenta actividad puntual en el primer corte semanal (~57%–58% sin interacción en el día 7, aunque parte de ellos interactúa en los días 14 o 21). 
* **Recomendación de investigación:** En lugar de asumir como hecho confirmado un abandono definitivo en la primera semana, se propone como **hipótesis prioritaria de producto** auditar la experiencia de *onboarding* durante los primeros 7 días posteriores al registro, diferenciando entre usuarios inactivos temporales, usuarios recurrentes y bajas definitivas.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** La tasa de conversión es IGUAL entre la interfaz de checkout actual y la nueva UI. Cualquier diferencia se debe al azar.
   - **H₁ (Hipótesis alternativa):** La tasa de conversión es DIFERENTE entre la interfaz actual y la nueva UI.

**Test estadístico:** Prueba z para proporciones (z-test).

**Nivel de significancia alpha:** 0.05

In [32]:
# tu código aquí
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

# 1. Cargar el dataset exacto de la prueba A/B
url_experimento = 'https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv'
df_checkout = pd.read_csv(url_experimento)

# Validar distribución de los grupos
print("Distribución de usuarios por variante:")
print(df_checkout['variante'].value_counts())

# 2. Contar conversiones (éxitos) y total de observaciones por variante
conversiones = df_checkout.groupby('variante')['convirtio'].sum()
totales = df_checkout.groupby('variante')['convirtio'].count()

# Preparar listas para proportions_ztest
grupos = conversiones.index.tolist()
exitos = [conversiones[grupos[0]], conversiones[grupos[1]]]
observaciones = [totales[grupos[0]], totales[grupos[1]]]

# 3. Aplicar Prueba Z para proporciones
z_stat, p_value = proportions_ztest(exitos, observaciones)

print(f"\n--- Resultados de la Prueba Z ---")
print(f"Estadístico z: {z_stat:.4f}")
print(f"Valor p: {p_value:.4f}")

# 4. Interpretar el resultado
alpha = 0.05
if p_value < alpha:
    print("\nRechazamos la hipótesis nula: Hay una diferencia estadísticamente significativa en la tasa de conversión.")
else:
    print("\nNo rechazamos la hipótesis nula: No hay evidencia de una diferencia real; los cambios en la UI no generaron un impacto significativo.")

# Mostrar tasas reales
tasa_0 = exitos[0] / observaciones[0]
tasa_1 = exitos[1] / observaciones[1]
print(f"\nTasa de conversión Grupo '{grupos[0]}': {tasa_0:.2%}")
print(f"Tasa de conversión Grupo '{grupos[1]}': {tasa_1:.2%}")

Distribución de usuarios por variante:
variante
tratamiento    5035
control        4965
Name: count, dtype: int64

--- Resultados de la Prueba Z ---
Estadístico z: -0.8133
Valor p: 0.4161

No rechazamos la hipótesis nula: No hay evidencia de una diferencia real; los cambios en la UI no generaron un impacto significativo.

Tasa de conversión Grupo 'control': 15.69%
Tasa de conversión Grupo 'tratamiento': 16.29%


### Conclusión y Recomendación del Experimento A/B

* **Decisión estadística:** Con un valor p de 0.4161 (mayor al umbral de 0.05), no rechazamos la hipótesis nula.
* **Interpretación de negocio:** Aunque la nueva UI (tratamiento) presentó una conversión ligeramente mayor (16.29% frente al 15.69% del control), la prueba Z demuestra que esta diferencia no es estadísticamente significativa y muy probablemente se deba al azar.
* **Recomendación:** No se recomienda invertir recursos de desarrollo en implementar la nueva interfaz a todos los usuarios, ya que no genera un aumento real en las ventas. Para mejorar la conversión, los esfuerzos deben redirigirse a solucionar la fricción detectada en la pasarela de pagos (etapa "add_payment_info" del funnel).

### Conclusión y Recomendación del Experimento A/B --- Correcciones

* **Decisión estadística:** Con un estadístico $z = -0.8133$ y un valor $p = 0.4161$ (superior al nivel de significancia $\alpha = 0.05$), **no rechazamos la hipótesis nula ($H_0$)**.
* **Interpretación rigurosa:** En la muestra analizada, el grupo de tratamiento presentó una conversión del **16.29%** frente al **15.69%** del grupo control (+0.60 puntos porcentuales). El resultado de la prueba Z indica que **no contamos con evidencia estadística suficiente** para concluir que esta mejora observada refleje una diferencia real a nivel poblacional (lo cual no equivale a demostrar que el efecto sea exactamente cero, sino que con el tamaño muestral actual es indistinguible de la variabilidad aleatoria).
* **Recomendación de negocio:** Se recomienda **no realizar un despliegue general (rollout)** de la nueva interfaz basándose únicamente en este experimento. Antes de comprometer recursos de ingeniería, conviene evaluar el tamaño de muestra requerido para detectar incrementos menores al 1% con suficiente potencia estadística, y priorizar la investigación de fricciones en el paso `add_payment_info` identificado en el embudo.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

  ---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---